# April run — smoke test

Exercises **every stage of the April pipeline end-to-end at tiny scope**
(1 day, all three symbols, full predictor/class spec, 5 training epochs) so a
broken step fails here in minutes instead of hours into the real run.
Checks:

1. data discovery + per-(symbol, group) config construction (filter ON) —
   NVDA/INTC from `data/NVDA_INTC`, IBM from its own `data/IBM`; one config
   per training group (bivariate / multivariate), as in the full run
2. distributions stage → **11 `SEQ_DISTR_*` (10 bivariate + 1 multivariate
   joint) + 50 `CLS_DISTR_*_{cls}`** files per symbol (no CLS for the joint
   entry — the colleague defines none)
3. ensemble stage (v2) → **20 `ENS_TD_*`** files per symbol (3 bivariate +
   1 joint channel, 5 lengths × 4 classes)
4. training, in the same two shapes as the full run: a direct `train_model`
   call (the `pipeline train` path) for the first symbol's two models, then
   **all six (symbol, group) models through the GPU fan-out** — one per
   device, with an assertion that the jobs actually spread across the
   visible GPUs and an exact-filename check of every `WGHTS_*` deliverable

Everything runs in `outputs/april_smoke/` — the real `outputs/april/` and
`configs/april_*.yaml` are untouched. Expected runtime: ~10-20 min on the
compute box (dominated by featurizing 1 day per symbol; the fan-out
trainings run concurrently, not back to back).

This validates **plumbing, not numbers** — numerical correctness is covered
by the byte-equivalence suites (`verify_cls_v2`, `verify_multivariate_seq`,
`verify_ensemble_reference`, `verify_ensemble_v2`, `verify_against_baseline`).

In [ ]:
import shutil, sys, time, pathlib
ROOT = pathlib.Path.cwd()
for p in (ROOT, ROOT / "scripts", ROOT / "tests", ROOT / "TrainingDistributions"):
    sys.path.insert(0, str(p))

from run_april import (find_data_dir, april_dates, detect_pattern,
                       DIST_PREDICTORS, CLS_NAMES, TRAIN_GROUPS)
from pipeline.config import RunConfig

SYMBOLS = ["NVDA", "INTC", "IBM"]
SMOKE_EPOCHS = 5          # tiny scope, not the real 3000
SMOKE_SEED = 0            # reproducible so a smoke failure is repeatable
SMOKE = ROOT / "outputs" / "april_smoke"
shutil.rmtree(SMOKE, ignore_errors=True)

# each symbol resolves its own directory (NVDA/INTC share NVDA_INTC; IBM is
# separate); the day-glob only runs once per distinct directory
resolved = {s: find_data_dir(s) for s in SYMBOLS}
scope_by_dir = {}
for s, data_dir in resolved.items():
    if data_dir not in scope_by_dir:
        pattern = detect_pattern(data_dir)
        dates = april_dates(data_dir, pattern)[:1]          # 1 day = tiny scope
        scope_by_dir[data_dir] = (pattern, dates)
        print(f"data: {data_dir}  (pattern: {pattern})  smoke day: {dates}")


def smoke_config(symbol, group):
    """Same shape as run_april.make_config, redirected to the smoke dir."""
    data_dir = resolved[symbol]
    pattern, dates = scope_by_dir[data_dir]
    cfg = RunConfig()
    cfg.data.symbol = symbol
    cfg.data.data_path = str(data_dir)
    cfg.data.dates = list(dates)
    cfg.data.file_pattern = pattern
    cfg.data.instrument_filter = True
    cfg.distributions.predictors = list(DIST_PREDICTORS)
    cfg.distributions.class_names = list(CLS_NAMES)
    cfg.distributions.class_values = [-1, 0, 1]
    cfg.distributions.output_dir = str((SMOKE / symbol).relative_to(ROOT))
    cfg.featurize.cache_dir = str((SMOKE / symbol / "feature_cache").relative_to(ROOT))
    cfg.ensemble.output_dir = str((SMOKE / symbol / "ensemble").relative_to(ROOT))
    cfg.training.model_dir = str((SMOKE / symbol / "models").relative_to(ROOT))
    # the group's hyperparameters, exactly as the full run writes them
    cfg.training.n_qubits = group["n_qubits"]
    cfg.training.batch_size = group["batch_size"]
    cfg.training.optimizer = group["optimizer"]
    cfg.training.max_seq_len = group["max_seq_len"]
    cfg.training.predictor_abbrev = group["predictor_abbrev"]
    # smoke scope, set BEFORE saving so the persisted YAML describes the run
    # that actually executes (CLAUDE.md rule 4) rather than the full-scale
    # defaults -- later cells must not override these afterwards.
    # One model per group: the group's FIRST predictor (for the multivariate
    # group that is already its only model). The fan-out sweeps
    # training.predictors (the list); without it the config would fall back
    # to all 11 distribution predictors at smoke scope.
    cfg.training.predictor = group["predictors"][0]
    cfg.training.predictors = [group["predictors"][0]]
    cfg.training.epochs = SMOKE_EPOCHS
    cfg.training.seed = SMOKE_SEED
    (SMOKE / symbol).mkdir(parents=True, exist_ok=True)
    cfg.save(SMOKE / f"{symbol.lower()}{group['suffix']}.yaml")
    return cfg

configs = {s + g["suffix"]: smoke_config(s, g)
           for s in SYMBOLS for g in TRAIN_GROUPS}
dist_configs = {s: configs[s] for s in SYMBOLS}   # group 0 runs stage 2 once
print(f"PASS  {len(configs)} configs built (filter ON, "
      f"{len(DIST_PREDICTORS)} predictors incl. the joint entry, "
      f"{len(CLS_NAMES)} classes, {len(TRAIN_GROUPS)} training groups)")

In [ ]:
from pipeline.runner import run

for symbol, cfg in dist_configs.items():
    t0 = time.time()
    run(cfg, run_id=f"smoke-dist-{symbol}")
    out = SMOKE / symbol
    preds = cfg.distributions.predictors
    exp_seq = len(preds)                                 # incl. the joint entry
    n_bi = len([p for p in preds if isinstance(p, str)])
    exp_cls = n_bi * len(cfg.distributions.class_names)  # strings only — the
                                                         # colleague defines no
                                                         # multivariate CLS
    pred = cfg.distributions.predicted
    n_seq = len(list(out.glob("SEQ_DISTR_*")))
    n_mv = len(list(out.glob(f"SEQ_DISTR_{symbol}_multivariate_*")))
    n_cls = len(list(out.glob("CLS_DISTR_*")))
    v2_named = len(list(out.glob(f"CLS_DISTR_{symbol}__{pred}-*_202504_*")))
    assert n_seq == exp_seq, f"{symbol}: expected {exp_seq} SEQ, got {n_seq}"
    assert n_mv == len(preds) - n_bi, f"{symbol}: multivariate SEQ missing"
    assert n_cls == exp_cls, f"{symbol}: expected {exp_cls} CLS, got {n_cls}"
    assert v2_named == exp_cls, f"{symbol}: CLS files not in v2 naming"
    print(f"PASS  {symbol} distributions: {n_seq} SEQ (incl. {n_mv} "
          f"multivariate) + {n_cls} CLS (v2 names) in {time.time()-t0:.0f}s  "
          f"[expected counts derived from the config]")

In [ ]:
from pipeline.ensemble import run_ensemble

# once per symbol — the group configs share every ensemble setting
for symbol, cfg in dist_configs.items():
    t0 = time.time()
    outputs = run_ensemble(cfg, run_id=f"smoke-ens-{symbol}")
    exp_ens = len(cfg.ensemble.seq_lengths) * len(cfg.ensemble.class_names)
    n_ch = len(cfg.ensemble.predictors)
    # file suffix follows the vendored reference the config selects
    suffix = "ALL" if cfg.ensemble.reference == "v2" else f"ALL_{n_ch}"
    n_ens = len(list((SMOKE / symbol / "ensemble").glob(f"ENS_TD_*_{suffix}")))
    assert n_ens == exp_ens, f"{symbol}: expected {exp_ens} ENS_TD, got {n_ens}"
    print(f"PASS  {symbol} ensemble ({cfg.ensemble.reference}): {n_ens} ENS_TD "
          f"files ({suffix} names) in {time.time()-t0:.0f}s  "
          f"[expected counts derived from the config]")

In [ ]:
from pipeline.models import train_model

# Direct-call check of the registered trainer (the `pipeline train` code
# path): ONE symbol's two models, sequentially on the default device. This
# cell validates the code path, not throughput — all six models are trained
# again through the GPU fan-out in the next cell, one per GPU, which is the
# path the full run actually uses and where the box's parallelism (and the
# exact-filename check for every symbol) is exercised.
for key in (SYMBOLS[0], SYMBOLS[0] + TRAIN_GROUPS[1]["suffix"]):
    # every training parameter already comes from the saved config; nothing
    # is overridden here, so the printed/saved YAML is what ran
    cfg = configs[key]
    r = train_model(cfg, run_id=f"april-smoke-train-{key}")
    assert pathlib.Path(r["model_file"]).exists()
    assert pathlib.Path(r["weights_file"]).exists()
    assert len(r["plots"]) == 4, f"expected 4 charts, got {len(r['plots'])}"
    print(f"PASS  {key} training: 2 result files + 4 charts "
          f"({r['n_sequences']} examples, {r['train_seconds']:.0f}s)")

print("\nTrainer OK (direct call). Fan-out trains all six across GPUs next.")

### 5 — stage-3 fan-out (the way `april_run.ipynb` actually trains)
The cell above calls `train_model` directly for one symbol. The real run instead
goes through `plan_training` -> `run_training_jobs`, which reads the schedule from
the config and dispatches one training per device in a spawned worker pool. That
path is what full scale uses, so it is exercised here at smoke scope: same 5
epochs, all six (symbol, group) models — re-training the first symbol's two over
the top and training the other four fresh, spread across the visible GPUs.

Checks the parts only the fan-out has: schedule read from
`training.gpus`/`training.max_parallel`, one job per (symbol, group), jobs
actually spread over the expected number of distinct devices, results arriving
in completion order, each job's charts surviving the spawned process — and,
once every model exists, the exact `WGHTS_*` deliverable filenames.

In [ ]:
import os
from run_april import plan_training, run_training_jobs, visible_gpu_ids

smoke_cfgs = {k: SMOKE / f"{k.lower()}.yaml" for k in configs}
jobs, n_par = plan_training(smoke_cfgs)        # schedule from the CONFIG
print(f"{len(jobs)} jobs | {n_par} at a time")
cvd = os.environ.get("CUDA_VISIBLE_DEVICES")
if cvd is not None:
    # the single most common cause of "everything ran on one GPU": torch
    # only sees (and renumbers) the devices this names, and training.gpus
    # 'auto' deliberately schedules within it. Unset it in the shell that
    # launches JupyterLab (and restart the kernel) to use the whole box.
    print(f"NOTE: CUDA_VISIBLE_DEVICES={cvd!r} restricts this run — "
          f"torch sees only these GPUs, renumbered from cuda:0")
for j in jobs:
    print(f"    {j['symbol']:6s} x {j['label']:16s} -> {j['device']}")
assert len(jobs) == len(configs), jobs         # 1 model per (symbol, group)
assert all(isinstance(j["label"], str) for j in jobs)
assert len({j["run_id"] for j in jobs}) == len(jobs), "run_ids must be unique"

# the jobs must actually spread over the GPUs the config selects — on the
# 8-GPU box these 6 jobs must land on 6 distinct devices, not queue on one
n_gpus = len(visible_gpu_ids(next(iter(configs.values())).training.gpus))
exp_devices = min(n_gpus, len(jobs)) or 1      # 0 GPUs -> everything on cpu
assert len({j["device"] for j in jobs}) == exp_devices, (
    f"jobs use {sorted({j['device'] for j in jobs})}, "
    f"expected {exp_devices} distinct device(s)")

seen = []
for r in run_training_jobs(jobs, n_par):       # spawned workers
    seen.append((r["symbol"], r["predictor"]))
    assert pathlib.Path(r["model_file"]).exists(), r
    assert pathlib.Path(r["weights_file"]).exists(), r
    assert not r.get("chart_error"), r["chart_error"]
    assert len(r["plots"]) == 4, f"{r['symbol']}: {len(r['plots'])} charts"
    print(f"PASS  {r['symbol']} x {r['predictor']} on {r['device']} "
          f"({r['train_seconds']:.0f}s)")

assert len(seen) == len(jobs), f"{len(seen)} results for {len(jobs)} jobs"
assert len(set(seen)) == len(seen), f"duplicate results: {seen}"

# the deliverable names, exactly as the colleague's drivers write them —
# checked after the fan-out so every (symbol, group) pair is covered
# (his AAPL runs produced e.g. WGHTS_AAPL_bivariate_log_mid-vpin_202504_3q.pt)
for s in SYMBOLS:
    models = SMOKE / s / "models"
    for name in (f"WGHTS_{s}_bivariate_log_mid-vpin_202504_3q.pt",
                 f"WGHTS_{s}_multivariate_log_mid-L10_micro_vpin_202504_6q.pt"):
        assert (models / name).exists(), f"missing {name} in {models}"
    print(f"PASS  {s}: WGHTS_* files carry the colleague's exact names")

print("\nFAN-OUT OK — plan_training + run_training_jobs work end to end.")
print("Safe to run april_run.ipynb at full scope.")